In [1]:
from fealpy.backend import backend_manager as bm
from fealpy.backend import TensorLike

from fealpy.mesh import UniformMesh2d
from fealpy.functionspace.lagrange_fe_space import LagrangeFESpace
from fealpy.decorator import cartesian
from fealpy.fem import BilinearForm, ScalarDiffusionIntegrator
from fealpy.fem import LinearForm, ScalarSourceIntegrator
from fealpy.fem import DirichletBC
from fealpy.solver import cg

import numpy as np


In [2]:
data = np.load("kf_quad.npz")

sample_id = 0

points = data["points"]
weights = data["quad_weights"]

k = data["k"][sample_id]
f = data["f"][sample_id]

print("points:", points.shape)
print("weights:", weights.shape)
print("k:", k.shape)
print("f:", f.shape)

points: (10000, 16, 2)
weights: (16,)
k: (10000, 16)
f: (10000, 16)


In [3]:
device = 'cpu'

bm.set_backend('pytorch')
bm.set_default_device(device)
dtype = bm.float32

In [4]:
k = bm.tensor(k, dtype=dtype, device=device)
f = bm.tensor(f, dtype=dtype, device=device)

In [5]:
class Exp1():
    def __init__(self, dtype = bm.float32):
        self.domain = [0, 1, 0, 1]
        self.dtype = dtype
    
    @cartesian
    def dirichlet(self, p: TensorLike) -> TensorLike:
        """Dirichlet boundary condition"""
        x = p[..., 0]
        return bm.tensor(x.shape, self.dtype)

In [6]:
PDE = Exp1()

domain = PDE.domain
nx, ny = 100, 100

hx = (domain[1] - domain[0])/nx
hy = (domain[3] - domain[2])/ny

mesh = UniformMesh2d((0, nx, 0, ny), h=(hx, hy), origin=(domain[0], domain[2]), ftype=bm.float32)


In [7]:
space= LagrangeFESpace(mesh, p=1)
uh = space.function()
bform = BilinearForm(space)
DI = ScalarDiffusionIntegrator(k)
bform.add_integrator(DI)

lform = LinearForm(space)
SI = ScalarSourceIntegrator(f)
lform.add_integrator(SI)

A = bform.assembly()
F = lform.assembly()

A, F = DirichletBC(space, gd=PDE.dirichlet()).apply(A, F)

c:\Users\Rainbow\.conda\envs\eit\Lib\site-packages\torch\utils\_device.py:106: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


RuntimeError: expected scalar type Double but found Float